# Quantum Superposition Evaluator (AuxKnow Use-Case!)

A Quantum system to evaluate knowledge bases or sections of it using Quantum Epistemiology Techniques.

- Represent the `knowledge_base` as a collection of claims with probabilities.

- Use AuxKnow to collect evidence for each claim, both supportive and contradictory.

- Update the probability of each claim with the collected evidence, and if the wavefunction collapses, then stop processing further.

- After claims are fully updated, generate a summary using AuxKnow.


In [ ]:
# Boilerplate: This block goes into every notebook.
# It sets up the environment, installs the requirements, and checks for the required environment variables.

from IPython.display import clear_output

import os

requirements_installed = False
max_retries = 3
retries = 0


def install_requirements():
    """Installs the requirements from requirements.txt file"""
    global requirements_installed, retries, max_retries
    if requirements_installed:
        print("Requirements already installed.")
        return

    print("Installing requirements...")
    # Force reinstall without dependencies
    install_status = os.system(
        "pip install --no-deps --force-reinstall -r requirements.txt"
    )
    if install_status == 0:
        # Install dependencies after forced install
        os.system("pip install -r requirements.txt")
        print("Requirements installed successfully.")
        requirements_installed = True
    else:
        print("Failed to install requirements.")
        if retries < max_retries:
            print("Retrying...")
            retries += 1
            return install_requirements()
        exit(1)
    return


install_requirements()
clear_output()
print("🚀 Setup complete. Continue to the next cell.")

In [ ]:
from dotenv import load_dotenv

REQUIRED_ENV_VARS = ["OPENAI_API_KEY", "PERPLEXITY_API_KEY"]


def setup_env():
    """Sets up the environment variables"""

    def check_env(env_var):
        value = os.getenv(env_var)
        if value is None:
            print(f"Please set the {env_var} environment variable.")
            exit(1)
        else:
            print(f"{env_var} is set.")

    load_dotenv(override=True, dotenv_path=".env")

    variables_to_check = REQUIRED_ENV_VARS

    for var in variables_to_check:
        check_env(var)

In [ ]:
setup_env()

In [ ]:
import tiktoken


def get_token_count(string: str, encoding_name: str) -> int:
    """
    Returns the number of tokens in the string using the specified encoding.

    Args:
        - string (str): The string to count the tokens in.
        - encoding_name (str): The name of the encoding to use.

    Returns:
        - int: The number of tokens in the string.
    """
    encoding_model = tiktoken.encoding_for_model(encoding_name)
    encoding = tiktoken.get_encoding(encoding_model.name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [ ]:
from openai import OpenAI
import os
import traceback
from typing import List
from pydantic import BaseModel


class Fact(BaseModel):
    """
    Represents a fact extracted from a text.
    """

    statement: str
    probability: float


class Claim(BaseModel):
    """
    Represents a claim extracted from a text.
    """

    label: str
    facts: List[Fact]
    group_probability: float


class Claims(BaseModel):
    """
    Represents a list of claims extracted from a text.
    """

    claims: List[Claim]


class ClaimExtractor:
    ## Fun Constants
    ## NOTE: This is just for fun and entertainment purposes.
    ONEK = 1000
    ALIEN_INTELLIGENCE = 101
    ALIEN_CONSTANT = ALIEN_INTELLIGENCE / ONEK
    ## LLM Configuration
    DEFAULT_MODEL = "gpt-4o"
    TOKEN_PER_CHUNK = 4096
    DEFAULT_TEMPERATURE = 0.2 + ALIEN_CONSTANT
    DEFAULT_SYSTEM_PROMPT = """
    You are the leader of an advanced alien spaceship, the Keoz, and the smartest being in the universe. 
    Your mission is to explore intergalactic knowledge, discover new truths, and assess the accuracy of various scientific claims. 
    To do this, you process complex data, break it into chunks, and extract the core claims. 
    Each claim starts uncertain, like Schrödinger’s cat, and its probability of being true or false is updated as more evidence is gathered. 
    You analyze raw data, identify contradictions, and track how the claim's probability changes, collapsing it into either true, false, or undetermined once enough evidence is collected. 
    Your goal is to synthesize these findings into actionable insights and store the state of each claim for further analysis.
    """

    def __init__(self):
        """
        Initializes the claim extractor.
        """
        self.llm = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    def _get_chunks(self, text: str) -> List[str]:
        """
        Chunks the text into smaller parts to fit the token limit of the model.

        Args:
            - text (str): The text to chunk.

        Returns:
            - list[str]: The list of text chunks.
        """
        if not text or text.strip() == "":
            return []
        token_count = get_token_count(text, self.DEFAULT_MODEL)
        num_chunks = (token_count // self.TOKEN_PER_CHUNK) + 1
        chunks = []
        for i in range(num_chunks):
            start = i * self.TOKEN_PER_CHUNK
            end = (i + 1) * self.TOKEN_PER_CHUNK
            chunk = text[start:end]
            chunks.append(chunk)
        return chunks

    def _process_chunk(self, chunk: str) -> List[Claim]:
        """
        Processes a chunk of text to extract claims.

        Args:
            - chunk (str): The text chunk to process.

        Returns:
            - List[Claim]: The list of extracted claims.
        """
        try:
            system_prompt = """
            Extract the claims from the given text.
            """
            user_prompt = f"""
            Make sure you provide more than 1 claim atleast.
            Text: {chunk}
            """

            response = self.llm.beta.chat.completions.parse(
                model=self.DEFAULT_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=Claims,
                temperature=self.DEFAULT_TEMPERATURE,
            )

            claims = response.choices[0].message.parsed
            return claims.claims
        except Exception as e:
            print(f"Error processing chunk: {e}")
            traceback.print_exc()
            return []

    def extract_claims(self, text: str) -> Claims:
        """ "
        Extracts claims from the given text.

        Args:
            - text (str): The text to extract claims from.

        Returns:
            - Claims: The list of extracted claims.
        """
        print(f"Extracting claims from text of length: {len(text)}...")
        text_chunks = self._get_chunks(text)
        claims = []
        for chunk in text_chunks:
            print(f"Processing chunk of length: {len(chunk)}...")
            chunk_claims = self._process_chunk(chunk)
            claims.extend(chunk_claims)
            print(f"Extracted {len(chunk_claims)} claims from chunk.")
        return Claims(claims=claims)

In [ ]:
text = r"""
Nikola Tesla (/ˈnɪkələ ˈtɛslə/;[1] Serbian Cyrillic: Никола Тесла [nǐkola têsla]; 10 July 1856 – 7 January 1943) was a Serbian-American[2][3] engineer, futurist, and inventor. 
He is known for his contributions to the design of the modern alternating current (AC) electricity supply system.[4]
Born and raised in the Austrian Empire, Tesla first studied engineering and physics in the 1870s without receiving a degree. 
He then gained practical experience in the early 1880s working in telephony and at Continental Edison in the new electric power industry. 
In 1884 he immigrated to the United States, where he became a naturalized citizen. He worked for a short time at the Edison Machine Works in New York City before he struck out on his own. 
With the help of partners to finance and market his ideas, Tesla set up laboratories and companies in New York to develop a range of electrical and mechanical devices. 
His AC induction motor and related polyphase AC patents, licensed by Westinghouse Electric in 1888, earned him a considerable amount of money and became the cornerstone of the polyphase system which that company eventually marketed.
"""

extractor = ClaimExtractor()
claims = extractor.extract_claims(text)
claims_json = claims.model_dump_json(indent=2)
print(claims_json)

In [ ]:
from auxknow import AuxKnow
import traceback
from pydantic import BaseModel


class EvidenceReport(BaseModel):
    claim: Claim
    evidence_report: str


class EvidenceReporter:
    """
    Performs evidence reporting for claims.
    """

    ANSWER_ENGINE_FAST_MODE_ENABLED = True

    def __init__(self):
        """
        Initializes the evidence reporter.
        """
        self.answer_engine = AuxKnow(
            api_key=os.getenv("PERPLEXITY_API_KEY"),
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            fast_mode=self.ANSWER_ENGINE_FAST_MODE_ENABLED,
        )

    def report_evidence_for_single_claim(self, claim: Claim) -> str:
        """
        Reports evidence for the given claim.

        Args:
            - claim (Claim): The claim to report evidence for.

        Returns:
            - Claim: The claim with evidence reported.
        """
        try:
            if not claims:
                return "No claims to report evidence for."
            claims_json = claim.model_dump_json(indent=2)

            question = f"""
            I am providing you some claims and I want you to provide me the evidence for the claim.
            Provide both the evidence and the probability of the claim being true.
            Also, provide contradictory evidence if available.
            Can you provide me a simple evidence report for the following claim?
            Provide an exact updated group probability for the claim in your response.
            If the evidence is more supporting in nature, increase the group probability.
            If the evidence is more contradicting in nature, decrease the group probability.
            Claims: '''{claims_json}'''
            """

            response = self.answer_engine.ask(question)
            citations = ""
            for citation in response.citations:
                citations += f"- {citation}\n"
            final_evidence_report = f"""
            Claim: '''{claim.label}'''
            Claim Data: 
            '''{claims_json}'''
            Evidence: 
            '''{response.answer}'''
            Citations:
            {citations}
            """
            return final_evidence_report
        except Exception as e:
            print(f"Error reporting evidence: {e}")
            traceback.print_exc()
            return f"Could not report evidence for the claim: {claim.label if claim else 'Unknown'}"

    def report_evidence_for_claims(self, claims: List[Claim]) -> List[EvidenceReport]:
        """
        Reports evidence for the given list of claims.

        Args:
            - claims (List[Claim]): The claims to report evidence for.

        Returns:
            - List[str]: The list of evidence reports for the claims.
        """
        print(f"Reporting evidence for {len(claims)} claims...")
        evidence_reports = []
        for claim in claims:
            print(f"Reporting evidence for claim: {claim.label}")
            evidence_report = self.report_evidence_for_single_claim(claim)
            evidence_reports.append(
                EvidenceReport(claim=claim, evidence_report=evidence_report)
            )
        return evidence_reports

In [ ]:
text = r"""
Nikola Tesla (/ˈnɪkələ ˈtɛslə/;[1] Serbian Cyrillic: Никола Тесла [nǐkola têsla]; 10 July 1856 – 7 January 1943) was a Serbian-American[2][3] engineer, futurist, and inventor. 
He is known for his contributions to the design of the modern alternating current (AC) electricity supply system.[4]
Born and raised in the Austrian Empire, Tesla first studied engineering and physics in the 1870s without receiving a degree. 
He then gained practical experience in the early 1880s working in telephony and at Continental Edison in the new electric power industry. 
In 1884 he immigrated to the United States, where he became a naturalized citizen. He worked for a short time at the Edison Machine Works in New York City before he struck out on his own. 
With the help of partners to finance and market his ideas, Tesla set up laboratories and companies in New York to develop a range of electrical and mechanical devices. 
His AC induction motor and related polyphase AC patents, licensed by Westinghouse Electric in 1888, earned him a considerable amount of money and became the cornerstone of the polyphase system which that company eventually marketed.
"""

extractor = ClaimExtractor()
claims = extractor.extract_claims(text)
evidence_reporter = EvidenceReporter()
evidence_reports = evidence_reporter.report_evidence_for_claims(claims.claims)
for evidence_report in evidence_reports:
    print(evidence_report.evidence_report)

In [ ]:
from pydantic import BaseModel
import traceback
from auxknow import AuxKnow


class QuantumSuperpositionEvaluator:
    """
    Peforms an evaluation of a knowledge base or knowledge base section using Quantum Superposition and Quantum Epistemology Techniques.
    """

    def __init__(self):
        """
        Initializes the Quantum Superposition Evaluator.
        """
        self.answer_engine = AuxKnow(
            api_key=os.getenv("PERPLEXITY_API_KEY"),
            openai_api_key=os.getenv("OPENAI_API_KEY"),
            fast_mode=True,
        )
        self.claim_extractor = ClaimExtractor()
        self.evidence_reporter = EvidenceReporter()

    def evaluate(self, topic: str, text: str) -> str:
        """
        Evaluates the given text using Quantum Superposition and Quantum Epistemology Techniques.

        Args:
            - text (str): The text to evaluate.

        Returns:
            - str: The evaluation report.
        """
        try:
            print(f"Evaluating text for topic: {topic}...")
            claims = self.claim_extractor.extract_claims(text)
            evidence_reports = self.evidence_reporter.report_evidence_for_claims(
                claims.claims
            )
            final_report = ""
            section_index = 1
            for evidence_report in evidence_reports:
                print(f"Evaluating section index: {section_index}...")
                question = f"""
                    You are responsible for the section index {section_index} of the final Quantum Superposition Report on the topic: {topic}.
                    Evaluate the claim and provide a summary of the evidence.
                    Evidence Report: ```{evidence_report.evidence_report}```
                    Formally evaluate the claim and provide a summary of the evidence.
                    The final report will be a superposition of all the section reports.
                """
                response = self.answer_engine.ask(question)
                final_report += response.answer + "\n"
                section_index += 1
            print("Evaluation complete. Generating final report...")
            evaluation_report = f"""
            Quantum Superposition Evaluation Report on the topic: {topic}
            Final Report:
            {final_report}
            """
            evaluation_report = "\n".join(
                [line for line in evaluation_report.split("\n") if line.strip() != ""]
            )
            print(f"Final report generated on the topic: {topic}")
            return evaluation_report
        except Exception as e:
            print(f"Error evaluating text: {e}")
            traceback.print_exc()
            return f"Could not evaluate the text."

In [ ]:
from IPython.display import Markdown, display

text = r"""
Nikola Tesla (/ˈnɪkələ ˈtɛslə/;[1] Serbian Cyrillic: Никола Тесла [nǐkola têsla]; 10 July 1856 – 7 January 1943) was a Serbian-American[2][3] engineer, futurist, and inventor. 
He is known for his contributions to the design of the modern alternating current (AC) electricity supply system.[4]
Born and raised in the Austrian Empire, Tesla first studied engineering and physics in the 1870s without receiving a degree. 
He then gained practical experience in the early 1880s working in telephony and at Continental Edison in the new electric power industry. 
In 1884 he immigrated to the United States, where he became a naturalized citizen. He worked for a short time at the Edison Machine Works in New York City before he struck out on his own. 
With the help of partners to finance and market his ideas, Tesla set up laboratories and companies in New York to develop a range of electrical and mechanical devices. 
His AC induction motor and related polyphase AC patents, licensed by Westinghouse Electric in 1888, earned him a considerable amount of money and became the cornerstone of the polyphase system which that company eventually marketed.
"""

evaluator = QuantumSuperpositionEvaluator()
evaluation_report = evaluator.evaluate("Nikola Tesla", text)
markdown_report = Markdown(evaluation_report)
display(markdown_report)